In [1]:
# basic imports
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# model imports
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neighbors import KNeighborsClassifier

# metric imports
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import accuracy_score
from sklearn.metrics import mean_absolute_error

# model selection imports
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV

# preprocessing imports
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [3]:
wine_train = pd.read_parquet(
    "https://lab.cs307.org/wine/data/wine-train.parquet",
)
wine_test = pd.read_parquet(
    "https://lab.cs307.org/wine/data/wine-test.parquet",
)

In [4]:
# create X and y for train
X_train = wine_train.drop("quality", axis=1)
y_train = wine_train["quality"]

# create X and y for test
X_test = wine_test.drop("quality", axis=1)
y_test = wine_test["quality"]

In [5]:
X_train.shape

(4157, 12)

In [6]:
X_train.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,color
0,7.6,0.23,0.64,12.9,0.033,54.0,170.0,0.99800,3.00,0.53,8.8,white
1,NaN,0.75,0.01,2.2,0.059,11.0,18.0,0.99242,3.39,0.40,NaN,red
2,7.4,0.67,0.12,1.6,0.186,5.0,21.0,0.99600,3.39,0.54,9.5,red
3,6.4,0.18,0.74,NaN,0.046,54.0,168.0,0.99780,3.58,0.68,10.1,white
4,6.7,0.35,0.32,9.0,0.032,29.0,113.0,0.99188,3.13,0.65,12.9,white


In [12]:
#train a model 
num_features = ['fixed acidity', 'volatile acidity','citric acid','residual sugar','chlorides','free sulfur dioxide','total sulfur dioxide','density','pH','sulphates','alcohol']
cat_features = ['color']

#preprocessing for num features 
numeric_transformer = Pipeline(
    [
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())
    ]
)

#define preprocessing for categorical features
categorical_transformer = Pipeline(
    [
        ('imputer', SimpleImputer(strategy="most_frequent")),
        ('encoder', OneHotEncoder())
    ]
)

#create general preprocessor 
preprocessor = ColumnTransformer(
    [
        ('numeric', numeric_transformer, num_features),
        ('categorical', categorical_transformer, cat_features)
    ],
    remainder= 'drop',
)

#put everything tgt, build knn model 
knn = Pipeline(
    [
        ('preprocessor', preprocessor),
        ('classifier', KNeighborsClassifier()),
    ]
)

#fit (train) ur model 
knn.fit(X_train, y_train)

#make predictions on the test 
y_pred = knn.predict(X_test)

#get accuracy 
print(mean_absolute_error(y_test, y_pred))


0.5288461538461539


In [14]:
param_grid = {
    "classifier__n_neighbors": range(1,25),
    "classifier__p": [1, 2],
    "preprocessor__numeric__scaler": [None, StandardScaler()],
}
mod = GridSearchCV(
    knn,
    param_grid,
    cv=5,
    scoring="neg_mean_absolute_error",
)

mod.fit(X_train, y_train)
# predict on the test data
y_test_pred = mod.predict(X_test)

# calculate and print the test accuracy
test_mae = mean_absolute_error(y_test, y_test_pred)
print(f"Test MAE: {test_mae}")


/Users/harlowng/Desktop/cs307/.venv/lib/python3.13/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


Test MAE: 0.47596153846153844


In [15]:
from joblib import dump
dump(mod, "wine.joblib")

['wine.joblib']